# Pattern #1: Reflection - Agents That Think

**From-Scratch Implementation**

## Overview

The Reflection pattern implements self-critique and iterative improvement:

```
Draft → Critique → Revise → Final Answer
```

This pattern enables agents to:
- Generate an initial response (Draft)
- Critically evaluate their own output (Critique)
- Improve based on self-reflection (Revise)

**No tools or RAG** - Pure agent cognition and self-improvement.


In [ ]:
import sys
sys.path.append('..')

from utils import create_llm_provider, get_config

# Initialize
config = get_config()
llm = create_llm_provider()

print(f"Using model: {config.get('model')}")
print(f"Temperature: {config.get('temperature')}")


### Validation (config from config/, default OpenRouter)

In [ ]:
assert config.get("model"), "config.get('model') should be set"
assert llm is not None, "create_llm_provider() should return a provider"
print("✓ Setup valid: config and LLM ready.")

## Reflection Functions

We'll implement three stages:
1. **Draft** - Generate initial response
2. **Critique** - Evaluate the draft
3. **Revise** - Improve based on critique


In [ ]:
def draft(query: str) -> dict:
    """Generate initial draft response."""
    system_prompt = """
You are a helpful healthcare assistant providing information about preparing for medical visits.
Generate a clear, helpful response to the user's question.
Focus on practical advice and common preparation steps.
""".strip()
    
    response = llm.generate(prompt=query, system_prompt=system_prompt)
    
    return {
        "draft": response["response"],
        "tokens": response["total_tokens"],
        "latency_ms": response["latency_ms"]
    }


def critique(query: str, draft_text: str) -> dict:
    """Critique the draft response."""
    system_prompt = """
You are a critical reviewer evaluating healthcare information responses.
Review the draft and provide constructive feedback on:
1. Completeness 2. Accuracy 3. Clarity 4. Actionability 5. Safety
Provide specific suggestions for improvement.
""".strip()
    
    prompt = f"""
Original Question: {query}

Draft Response: {draft_text}

Please provide a critical review with specific improvement suggestions.
""".strip()
    
    response = llm.generate(prompt=prompt, system_prompt=system_prompt)
    
    return {
        "critique": response["response"],
        "tokens": response["total_tokens"],
        "latency_ms": response["latency_ms"]
    }


def revise(query: str, draft_text: str, critique_text: str) -> dict:
    """Revise draft based on critique."""
    system_prompt = """
You are a healthcare assistant revising your response based on feedback.
Improve your draft by addressing all critique points.
Always end with: "This is educational information; verify with the hospital / consult your clinician."
""".strip()
    
    prompt = f"""
Original Question: {query}

Your Draft: {draft_text}

Critique Feedback: {critique_text}

Please provide an improved response that addresses all feedback points.
""".strip()
    
    response = llm.generate(prompt=prompt, system_prompt=system_prompt)
    
    return {
        "revised": response["response"],
        "tokens": response["total_tokens"],
        "latency_ms": response["latency_ms"]
    }


def assess_critique_sufficient(query: str, draft_text: str, critique_text: str) -> dict:
    """
    Use the LLM to assess whether the critique indicates the draft is already
    of sufficient quality (no meaningful improvements needed). Avoids hardcoded
    phrase matching in favor of semantic judgment.
    
    Returns:
        dict with keys: sufficient (bool), reasoning (str), tokens, latency_ms
    """
    system_prompt = """
You are an evaluator judging whether a critique of a healthcare response indicates
that the draft is already of sufficient quality and needs no further revision.

Consider: Does the critique suggest only minor/nitpicky improvements, or does it
identify substantive gaps (missing info, inaccuracies, clarity issues, safety gaps)?
If the critique is largely positive and suggests no major improvements, the draft
is sufficient. If it clearly asks for important additions or corrections, it is not.

Respond in exactly this format (no other text):
SUFFICIENT: YES or NO
REASONING: <one short sentence explaining your judgment>
""".strip()
    
    prompt = f"""
Original question: {query}

Draft response (excerpt): {draft_text[:500]}{"..." if len(draft_text) > 500 else ""}

Critique: {critique_text}

Is the draft already sufficient based on this critique? Use SUFFICIENT and REASONING as specified.
""".strip()
    
    response = llm.generate(prompt=prompt, system_prompt=system_prompt)
    text = response["response"].strip().upper()
    
    sufficient = "SUFFICIENT: YES" in text or "SUFFICIENT:YES" in text
    reasoning = ""
    if "REASONING:" in response["response"]:
        reasoning = response["response"].split("REASONING:")[-1].strip().split("\n")[0]
    
    return {
        "sufficient": sufficient,
        "reasoning": reasoning or ("Critique indicates sufficient quality." if sufficient else "Critique suggests further improvement needed."),
        "tokens": response["total_tokens"],
        "latency_ms": response["latency_ms"]
    }


def reflection_loop(query: str, max_iterations: int = 1, verbose: bool = True) -> dict:
    """
    Run complete Draft → Critique → Revise loop with optional multiple iterations.
    
    Args:
        query: User question
        max_iterations: Number of critique-revise cycles (default: 1)
        verbose: Print intermediate steps
    
    Returns:
        Complete results with all iterations and metrics
    """
    import time
    
    start_time = time.time()
    
    # Stage 1: Initial Draft
    if verbose:
        print("=" * 80)
        print("STAGE 1: INITIAL DRAFT")
        print("=" * 80)
    
    draft_result = draft(query)
    current_response = draft_result["draft"]
    total_tokens = draft_result["tokens"]
    
    if verbose:
        print(f"\n{current_response}")
        print(f"\n[Tokens: {draft_result['tokens']}, Latency: {draft_result['latency_ms']}ms]\n")
    
    # Store all iterations
    iterations = []
    
    # Stage 2: Iterative Critique → Revise
    for i in range(max_iterations):
        if verbose:
            print("=" * 80)
            if max_iterations == 1:
                print("STAGE 2: CRITIQUE")
            else:
                print(f"ITERATION {i + 1} of {max_iterations}: CRITIQUE")
            print("=" * 80)
        
        critique_result = critique(query, current_response)
        total_tokens += critique_result["tokens"]
        
        if verbose:
            print(f"\n{critique_result['critique']}")
            print(f"\n[Tokens: {critique_result['tokens']}, Latency: {critique_result['latency_ms']}ms]\n")
        
        # Assess whether critique indicates quality is sufficient (for multi-iteration mode)
        # Uses LLM-based judgment instead of hardcoded phrase matching
        if max_iterations > 1:
            assessment = assess_critique_sufficient(query, current_response, critique_result["critique"])
            total_tokens += assessment.get("tokens", 0)
            if assessment.get("sufficient", False):
                if verbose:
                    print(f"✓ Quality sufficient (assessor: {assessment.get('reasoning', 'N/A')}). Stopping at iteration {i + 1}.")
                iterations.append({
                    "iteration": i + 1,
                    "critique": critique_result,
                    "revised": None,
                    "stopped_early": True,
                    "assessment": assessment
                })
                break
        
        # Stage 3: Revise
        if verbose:
            print("=" * 80)
            if max_iterations == 1:
                print("STAGE 3: REVISE")
            else:
                print(f"ITERATION {i + 1} of {max_iterations}: REVISE")
            print("=" * 80)
        
        revise_result = revise(query, current_response, critique_result["critique"])
        total_tokens += revise_result["tokens"]
        current_response = revise_result["revised"]
        
        if verbose:
            print(f"\n{current_response}")
            print(f"\n[Tokens: {revise_result['tokens']}, Latency: {revise_result['latency_ms']}ms]\n")
        
        iterations.append({
            "iteration": i + 1,
            "critique": critique_result,
            "revised": revise_result,
            "stopped_early": False
        })
    
    end_time = time.time()
    total_time_ms = int((end_time - start_time) * 1000)
    
    if verbose:
        print("=" * 80)
        print("SUMMARY")
        print("=" * 80)
        if max_iterations == 1:
            print(f"Total tokens: {total_tokens}")
        else:
            print(f"Initial draft + {len(iterations)} iteration(s)")
            print(f"Total tokens: {total_tokens}")
            print(f"Avg tokens per iteration: {total_tokens // (len(iterations) + 1)}")
        print(f"Total time: {total_time_ms}ms")
    
    return {
        "query": query,
        "draft": draft_result,
        "iterations": iterations,
        "final_answer": current_response,
        "total_iterations": len(iterations),
        "total_tokens": total_tokens,
        "total_time_ms": total_time_ms
    }


## Example 1: Preparing for Dermatology Visit


In [ ]:
query1 = "How should I prepare for a dermatology OPD visit?"
result1 = reflection_loop(query1, verbose=True)


## Example 2: Same-Day Consultation


The `reflection_loop` now supports multiple iterations! Set `max_iterations` > 1 for progressive refinement.

**Pattern**: Draft → [Critique → Revise] × N

Benefits:
- Progressive quality improvement
- Catches issues missed in earlier critiques
- Automatic early stopping when quality is sufficient
- Similar to human writers revising multiple drafts


In [ ]:
query2 = "What precautions should I take before blood tests?"

# Run with 3 iterations for progressive refinement
result2 = reflection_loop(query2, max_iterations=3, verbose=True)


## Pattern Summary

The Reflection pattern demonstrates pure agent **cognition** without external tools:

**Benefits:**
- More complete and accurate responses
- Self-correction of mistakes
- Better structured outputs
- Safety-aware (ensures disclaimers)
- **Multi-iteration support** for progressive refinement

**Usage:**
```python
# Single iteration (default)
result = reflection_loop(query)

# Multiple iterations for higher quality
result = reflection_loop(query, max_iterations=3)
```

**Trade-offs:**
- More LLM calls (higher cost)
- Higher latency
- More tokens consumed
- Diminishing returns after 2-3 iterations

**When to use:**
- High-stakes outputs (medical, legal, etc.)
- Complex queries requiring nuanced answers
- When accuracy > speed
- Use max_iterations=2-3 for critical content

**Smart Features:**
- Automatic early stopping when quality is sufficient
- Tracks all iterations with metrics
- Backward compatible (defaults to 1 iteration)

The Reflection pattern is the foundation for more advanced agentic architectures.
